# Factor analysis

_Classical ML_

**Explain many measured signals with a few hidden causes plus noise.**

Factor analysis says your many observed variables are driven by a few hidden ones.

     Each hidden factor influences several observed signals at once.

---

This notebook builds factor analysis up one step at a time: first generate data from known hidden factors, then watch scikit-learn recover them, then read off how those factors load onto real wine chemistry. Run each cell, read the note above it, then experiment. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

Factor analysis assumes each observed signal is a weighted mix of a few hidden **factors** plus its own noise. To see it work, we first *manufacture* data that truly has this structure, then ask the model to recover it.

### Step 1 — Generate signals from known hidden factors

We draw `n` examples, each with `k=2` hidden factors `Z`. A fixed **loading matrix** `Lam` maps those 2 factors onto `p=6` observed signals, and we add independent per-variable noise. Because *we* built the data with 2 true factors, we know the right answer the model should find.

In [ ]:
import numpy as np
from sklearn.decomposition import FactorAnalysis

rng = np.random.default_rng(0)

n = 500   # number of examples
k = 2     # number of hidden factors (the ground truth)
p = 6     # number of observed signals per example

Z = rng.normal(size=(n, k))                 # hidden factors, one row per example
Lam = rng.normal(size=(k, p))               # loadings: how factors map to signals
noise = rng.normal(0, 0.5, size=(n, p))     # independent per-variable noise

X = Z @ Lam + noise                         # the observed signals we actually see
print("observed data shape:", X.shape)

### Step 2 — Fit a 2-factor model and inspect it

We fit `FactorAnalysis` with `n_components=2`, matching the true number of factors. The learned `components_` is the recovered loading matrix, and `noise_variance_` is the per-signal noise it estimated. The `score` is the average log-likelihood — higher means the model explains the data better.

In [ ]:
fa = FactorAnalysis(n_components=2, random_state=0).fit(X)

print("loading matrix shape:", fa.components_.shape)
print("learned noise variance per signal:",
      np.round(fa.noise_variance_, 2))
print("avg log-likelihood:", round(fa.score(X), 3))

### Step 3 — Confirm 2 factors beats 1

Since the data was built with 2 true factors, a model allowed only 1 factor should fit **worse**. We fit a 1-factor model and compare its log-likelihood score against the 2-factor model — the 2-factor score should be higher, confirming the model has detected the real structure.

In [ ]:
# A 1-factor model is too simple for data with 2 true factors.
fa1 = FactorAnalysis(n_components=1, random_state=0).fit(X)

print("score with 1 factor :", round(fa1.score(X), 3))
print("score with 2 factors:", round(fa.score(X), 3))

## Visualize the data & results

_On real wine chemistry, how do two hidden factors load onto the measured signals?_

### Step 1 — Load and standardize six wine chemistry signals

We switch to 178 real wines. We pick six interpretable chemistry measurements and **standardize** them so each signal has comparable scale — otherwise a large-magnitude feature like proline would dominate the factors purely because of its units.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis

wine = load_wine()                       # 178 real wines, 13 chemistry features

idx = [0, 1, 5, 6, 9, 12]                # columns we keep
names = ["alcohol", "malic acid", "phenols", "flavanoids", "color", "proline"]

X = StandardScaler().fit_transform(wine.data)[:, idx]
print("wine signal matrix shape:", X.shape)

### Step 2 — Fit factor analysis and read the loadings

We fit a 2-factor model to the standardized wine signals. The `components_` matrix has shape `(2, 6)`: each row is one hidden factor, and each entry says how strongly that factor loads onto a given chemistry signal. A large positive or negative value means that signal is heavily driven by that factor.

In [ ]:
fa = FactorAnalysis(n_components=2, random_state=0).fit(X)

loadings = fa.components_                 # shape (2, 6): one row per factor
print("loadings shape:", loadings.shape)

### Step 3 — Plot how each factor loads onto the signals

We draw one bar chart per factor. Reading the bars tells you what each hidden factor *means* in chemistry terms — for example, a factor that loads heavily on phenols and flavanoids is capturing a shared 'richness' signal behind those measurements.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# Factor 1 loadings
ax[0].bar(names, loadings[0], color="#4ea1ff")
ax[0].set_title("Factor 1 loadings across six wine chemistry signals")

# Factor 2 loadings
ax[1].bar(names, loadings[1], color="#c89bff")
ax[1].set_title("Factor 2 loadings across six wine chemistry signals")

for a in ax:
    a.tick_params(axis="x", rotation=45)

plt.show()